## Block 0 — Smart Stratified Sampling (Run Before Everything)
Shortlists your full dataset to **~3,500 cases** while guaranteeing every `keyword_group` tag has **10–20 cases minimum**.

**How it works:**
1. First pass — collects minimum quota (10 cases) for every keyword that has fewer than 10 in the full data
2. Second pass — fills remaining slots up to 3,500 by sampling proportionally from all keywords
3. Rare keywords get their quota protected; dominant keywords get capped
4. Saves shortlisted file to `saved_data/shortlisted_3500.csv` — used as `DATA_FILE` in Block 2

> Set `RAW_FILE` to your original full CSV. Block 2's `DATA_FILE` is automatically updated.


In [ ]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

# ── CONFIGURATION ─────────────────────────────────────────────
RAW_FILE        = 'cfpb_complaints_simulated.csv'   # <-- YOUR FULL RAW CSV
TARGET_N        = 3500    # total cases to keep
MIN_PER_KEYWORD = 10      # guaranteed minimum per keyword
MAX_PER_KEYWORD = 20      # soft cap for rare keywords (prevents over-quota)
RANDOM_SEED     = 42
Path('saved_data').mkdir(exist_ok=True)

# ── LOAD ──────────────────────────────────────────────────────
df_raw = pd.read_csv(RAW_FILE)
print(f"Raw file: {len(df_raw):,} rows  |  {df_raw.shape[1]} columns")

# ── PARSE keyword_group ────────────────────────────────────────
def parse_kw(val):
    if isinstance(val, list): return [str(x).strip() for x in val if str(x).strip()]
    if isinstance(val, str):
        val = val.strip()
        if val.startswith('['):
            try: return [str(x).strip() for x in ast.literal_eval(val) if str(x).strip()]
            except: pass
        return [x.strip() for x in val.split(',') if x.strip()]
    return []

df_raw['_kw_parsed'] = df_raw['keyword_group'].apply(parse_kw)

# ── BUILD KEYWORD → ROW INDEX MAP ────────────────────────────
from collections import defaultdict
kw_to_idx = defaultdict(list)
for i, kws in enumerate(df_raw['_kw_parsed']):
    for kw in kws:
        kw_to_idx[kw].append(i)

all_keywords = sorted(kw_to_idx.keys())
print(f"\nUnique keywords found: {len(all_keywords)}")
kw_counts = {kw: len(idxs) for kw, idxs in kw_to_idx.items()}
kw_series  = pd.Series(kw_counts).sort_values(ascending=False)
print(f"\nKeyword frequency (top 10):")
print(kw_series.head(10).to_string())
print(f"\nKeyword frequency (bottom 10):")
print(kw_series.tail(10).to_string())
print(f"\nKeywords with < {MIN_PER_KEYWORD} cases: {(kw_series < MIN_PER_KEYWORD).sum()}")
print(f"Keywords with < 20 cases:           {(kw_series < 20).sum()}")

# ── PHASE 1: PROTECT RARE KEYWORDS ────────────────────────────
# For every keyword, collect up to MAX_PER_KEYWORD guaranteed cases
rng          = np.random.RandomState(RANDOM_SEED)
selected_idx = set()
quota_log    = {}

for kw in all_keywords:
    idxs     = kw_to_idx[kw]
    n_need   = min(MAX_PER_KEYWORD, max(MIN_PER_KEYWORD, len(idxs)))
    # Prioritise rows not yet selected (fresh quota)
    fresh    = [i for i in idxs if i not in selected_idx]
    already  = [i for i in idxs if i in selected_idx]
    n_fresh  = min(n_need, len(fresh))
    chosen   = rng.choice(fresh, size=n_fresh, replace=False).tolist() if fresh else []
    selected_idx.update(chosen)
    quota_log[kw] = len(chosen) + len([i for i in already])

print(f"\nPhase 1 (guaranteed quota): {len(selected_idx):,} rows selected")

# ── PHASE 2: FILL TO TARGET_N ─────────────────────────────────
# Fill remaining slots by proportional sampling from all keywords
remaining_slots = TARGET_N - len(selected_idx)
unselected_idx  = [i for i in df_raw.index if i not in selected_idx]

if remaining_slots > 0 and unselected_idx:
    # Weight unselected rows by inverse keyword frequency (rare = higher weight)
    row_weights = np.zeros(len(unselected_idx))
    for j, idx in enumerate(unselected_idx):
        kws = df_raw.iloc[idx]['_kw_parsed']
        if kws:
            # Use inverse frequency so rows from rare keywords fill first
            row_weights[j] = np.mean([1.0 / max(kw_counts.get(kw, 1), 1) for kw in kws])
        else:
            row_weights[j] = 1.0 / len(df_raw)  # uniform fallback

    row_weights = row_weights / row_weights.sum()   # normalise to probabilities
    n_to_sample = min(remaining_slots, len(unselected_idx))
    extra_idx   = rng.choice(unselected_idx, size=n_to_sample, replace=False, p=row_weights)
    selected_idx.update(extra_idx.tolist())
    print(f"Phase 2 (fill to {TARGET_N:,}): added {len(extra_idx):,} rows")

# ── BUILD FINAL SHORTLISTED DF ────────────────────────────────
df_short = df_raw.iloc[sorted(selected_idx)].drop(columns=['_kw_parsed']).reset_index(drop=True)
print(f"\nFinal shortlisted size: {len(df_short):,} rows")

# ── VERIFY KEYWORD COVERAGE ───────────────────────────────────
df_short['_kw_parsed'] = df_short['keyword_group'].apply(parse_kw)
kw_counts_short = defaultdict(int)
for kws in df_short['_kw_parsed']:
    for kw in kws:
        kw_counts_short[kw] += 1

coverage_df = pd.DataFrame({
    'keyword':       list(kw_counts_short.keys()),
    'count_full':    [kw_counts.get(k, 0) for k in kw_counts_short],
    'count_short':   list(kw_counts_short.values()),
}).sort_values('count_short', ascending=False)

below_min = coverage_df[coverage_df['count_short'] < MIN_PER_KEYWORD]

print(f"\nKeyword coverage in shortlisted set:")
print(coverage_df.to_string(index=False))
print(f"\nKeywords below {MIN_PER_KEYWORD} cases: {len(below_min)}", end="")
if len(below_min):
    print(f" (these had < {MIN_PER_KEYWORD} total in the full dataset — cannot fix):")
    print(below_min[['keyword','count_full','count_short']].to_string(index=False))
else:
    print("  -- All keywords meet minimum quota!")

# ── SAVE ──────────────────────────────────────────────────────
df_short = df_short.drop(columns=['_kw_parsed'])
out_path = 'saved_data/shortlisted_3500.csv'
df_short.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Set DATA_FILE = '{out_path}' in Block 2 to use this file")

# Auto-update DATA_FILE for the session
DATA_FILE = out_path
print(f"\nDATA_FILE updated to: {DATA_FILE}")


# Dynamic Complaint Tagger — Definitive Pipeline v4

**Run each block independently OR top-to-bottom as a full pipeline.**

| Block | What it does |
|---|---|
| 1 | Installs & imports |
| 2 | Load & parse data |
| 3 | Auto-diagnose & compute all settings |
| 4 | Drop rare tags |
| 5 | CleanLab noise detection (fast concatenated column method) |
| 6 | Train / Val / Test split + save |
| 7 | Imbalance handling (augment + undersample + weighted sampler) |
| 8 | Build input text + Dataset + DataLoaders |
| 9 | Model + Loss + Optimizer |
| 10 | Training loop with epoch checkpoints |
| 11 | Per-tag threshold tuning |
| 12 | Final test evaluation |
| 13 | Human review export |
| 14 | Feedback loop (after human review) |
| 15 | Inference on new complaints |

> **Only change needed:** Set `DATA_FILE` in Block 2 to your CSV filename.


## Block 1 — Installs & Imports

In [ ]:
import subprocess, sys
for pkg in ["transformers", "torch", "scikit-learn", "tqdm", "pandas", "numpy", "nbformat"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"])
try:
    import cleanlab
    print("cleanlab already installed")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "cleanlab", "-q"])
    print("cleanlab installed")


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import ast, random, os, json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'distilbert-base-uncased'

for folder in ['checkpoints', 'saved_data', 'human_review']:
    Path(folder).mkdir(exist_ok=True)

# ── Checkpoint helpers used across all blocks ──────────────────
def checkpoint_exists(name):
    return Path(f'checkpoints/{name}').exists()

def save_checkpoint(obj, name):
    p = f'checkpoints/{name}'
    if isinstance(obj, pd.DataFrame): obj.to_csv(p, index=False)
    elif isinstance(obj, np.ndarray): np.save(p, obj)
    elif isinstance(obj, dict):
        with open(p, 'w') as f: json.dump(obj, f)
    print(f"  Saved: {name}")

def load_checkpoint(name):
    p = f'checkpoints/{name}'
    if name.endswith('.csv'):  return pd.read_csv(p)
    if name.endswith('.npy'):  return np.load(p, allow_pickle=True)
    if name.endswith('.json'):
        with open(p) as f: return json.load(f)

print(f"Device: {DEVICE}  |  Model: {MODEL_NAME}")
print("All imports OK — checkpoints/ saved_data/ human_review/ created")


## Block 2 — Load & Parse Data
> **Change `DATA_FILE` to your CSV filename before running.**

In [ ]:
DATA_FILE = 'cfpb_complaints_simulated.csv'   # <-- YOUR FILE HERE

def parse_tags(val):
    if isinstance(val, list): return [t.strip() for t in val if str(t).strip()]
    if isinstance(val, str):
        val = val.strip()
        if val.startswith('['):
            try: return [t.strip() for t in ast.literal_eval(val) if str(t).strip()]
            except: pass
        return [t.strip() for t in val.split(',') if t.strip()]
    return []

def parse_list_str(val):
    if isinstance(val, list): return val
    if isinstance(val, str) and val.startswith('['):
        try: return ast.literal_eval(val)
        except: pass
    return [val] if isinstance(val, str) and val.strip() else []

df = pd.read_csv(DATA_FILE)
df['tags_parsed']          = df['tier_2'].apply(parse_tags)
df['keyword_group_parsed'] = df['keyword_group'].apply(parse_list_str)
df = df[df['tags_parsed'].apply(len) > 0].reset_index(drop=True)

mlb = MultiLabelBinarizer()
y   = mlb.fit_transform(df['tags_parsed'])
tag_counts_s    = pd.Series(y.sum(axis=0), index=mlb.classes_).sort_values(ascending=False)
n_total         = len(df)
imbalance_ratio = tag_counts_s.max() / max(tag_counts_s.min(), 1)
tags_per_row    = y.sum(axis=1)

print(f"Loaded: {n_total} complaints  |  {len(mlb.classes_)} unique tier_2 tags")
print(f"Avg tags/complaint: {tags_per_row.mean():.2f}  |  Imbalance ratio: {imbalance_ratio:.0f}x")
print(f"\nTop 5 tags:\n{tag_counts_s.head().to_string()}")
print(f"\nBottom 5 tags:\n{tag_counts_s.tail().to_string()}")


## Block 3 — Auto-Diagnose & Compute All Settings
Reads your data and auto-computes every hyperparameter. Nothing hardcoded.

In [ ]:
HAS_CLEANLAB = 'cleanlab_avg_quality' in df.columns
if HAS_CLEANLAB:
    for col in ['has_label_issue', 'cleanlab_has_issue']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()                             .map({'TRUE':True,'FALSE':False,'1':True,'0':False}).fillna(False)
    avg_quality = df['cleanlab_avg_quality'].fillna(0.9).mean()
    pct_flagged = df['has_label_issue'].mean() if 'has_label_issue' in df.columns else 0.0
    df['noise_score'] = (1.0 - df['cleanlab_avg_quality'].fillna(avg_quality).clip(0,1))
else:
    avg_quality, pct_flagged = 0.9, 0.0
    tc = y.sum(axis=1); exp = tc.mean(); std = tc.std() + 1e-6
    df['noise_score'] = (np.abs(tc - exp) / (std * 3)).clip(0, 0.5)

MIN_TAG_SAMPLES = max(3, min(10, int(n_total * 0.005)))
RARE_THRESHOLD  = max(10, int(n_total * 0.03))
DOMINANT_FREQ   = 0.50
RARE_FREQ       = max(0.02, min(0.08, 1.5 / len(mlb.classes_)))
KW_DROPOUT      = min(0.55, 0.20 + (min(imbalance_ratio, 100) / 200))
BASE_SMOOTH     = round(min(0.15, max(0.02, 0.02 + (1.0 - avg_quality) * 0.25)), 3)
GAMMA_NEG       = int(min(6, max(2, 2 + min(imbalance_ratio, 100) / 30)))
EPOCHS          = max(5, min(12, int(20000 / n_total + 3)))
BATCH_SIZE      = 32 if n_total >= 5000 else 16 if n_total >= 1000 else 8
EARLY_STOP_PAT  = 3 if n_total >= 2000 else 4
LR              = 2e-5
med_chars       = df['summary'].dropna().str.len().median() if 'summary' in df.columns else 600
MAX_LEN         = 512 if med_chars > 1200 else 384 if med_chars > 600 else 256
DO_AUGMENT      = bool((tag_counts_s < RARE_THRESHOLD).any())
DO_UNDERSAMPLE  = bool(imbalance_ratio > 5 and (tag_counts_s / n_total > DOMINANT_FREQ).any())

print("DATA DIAGNOSIS REPORT")
print("=" * 55)
print(f"  Total complaints:       {n_total}")
print(f"  Unique tier_2 tags:     {len(mlb.classes_)}")
print(f"  Avg tags/complaint:     {tags_per_row.mean():.2f}")
print(f"  Imbalance ratio:        {imbalance_ratio:.0f}x")
print(f"  Most common tag:        '{tag_counts_s.index[0]}' ({int(tag_counts_s.max())} samples)")
print(f"  Least common tag:       '{tag_counts_s.index[-1]}' ({int(tag_counts_s.min())} samples)")
print(f"  Tags < 5 samples:       {(tag_counts_s < 5).sum()}")
print(f"  Tags < {RARE_THRESHOLD} samples:      {(tag_counts_s < RARE_THRESHOLD).sum()} (will be augmented)")
print(f"  CleanLab available:     {HAS_CLEANLAB}")
print(f"  Avg label quality:      {avg_quality:.3f}")
print(f"  Pct flagged noisy:      {pct_flagged*100:.1f}%")
print()
print("  AUTO-COMPUTED SETTINGS:")
print(f"  MIN_TAG_SAMPLES={MIN_TAG_SAMPLES}  RARE_THRESHOLD={RARE_THRESHOLD}  KW_DROPOUT={KW_DROPOUT:.2f}")
print(f"  BASE_SMOOTH={BASE_SMOOTH}  GAMMA_NEG={GAMMA_NEG}  EPOCHS={EPOCHS}")
print(f"  BATCH_SIZE={BATCH_SIZE}  MAX_LEN={MAX_LEN}  LR={LR}")
print(f"  DO_AUGMENT={DO_AUGMENT}  DO_UNDERSAMPLE={DO_UNDERSAMPLE}")
print("=" * 55)


## Block 4 — Drop Rare Tags
Removes tier_2 tags with fewer than `MIN_TAG_SAMPLES` examples — too few to learn from reliably.

In [ ]:
valid_mask = y.sum(axis=0) >= MIN_TAG_SAMPLES
dropped    = mlb.classes_[~valid_mask]
valid_tags = mlb.classes_[valid_mask]
y          = y[:, valid_mask]
NUM_LABELS = len(valid_tags)

print(f"Kept: {NUM_LABELS} tags  |  Dropped: {len(dropped)} tags with < {MIN_TAG_SAMPLES} samples")
if len(dropped): print(f"Dropped tags: {list(dropped)}")

keep = y.sum(axis=1) > 0
df, y = df[keep].reset_index(drop=True), y[keep]
print(f"Complaints after filter: {len(df)}")


## Block 5 — CleanLab Noise Detection (Fast Concatenated Column Method)
**The smart way:** one concatenated text column -> TF-IDF (capped at 15k features) -> cross-val probabilities -> CleanLab quality scores.

- Runs in ~2-3 minutes on 3000 rows
- Cached to `checkpoints/cleanlab_scores.csv` — never reruns unless you delete it
- Falls back to tag-count consistency scoring if CleanLab is not installed

In [ ]:
CLEANLAB_CACHE = 'checkpoints/cleanlab_scores.csv'

if HAS_CLEANLAB:
    print("CleanLab scores already in data — skipping computation")

elif checkpoint_exists('cleanlab_scores.csv'):
    print("Loading cached CleanLab scores...")
    cl_scores = load_checkpoint('cleanlab_scores.csv')
    df['cleanlab_avg_quality'] = cl_scores['cleanlab_avg_quality'].values
    df['cleanlab_has_issue']   = cl_scores['cleanlab_has_issue'].values
    df['noise_score']          = (1.0 - df['cleanlab_avg_quality'].clip(0,1))
    avg_quality = df['cleanlab_avg_quality'].mean()
    BASE_SMOOTH = round(min(0.15, max(0.02, 0.02 + (1.0 - avg_quality) * 0.25)), 3)
    print(f"  Avg quality: {avg_quality:.3f}  |  BASE_SMOOTH updated to: {BASE_SMOOTH}")

else:
    print("Running CleanLab — fast concatenated-column method...")

    # SMART WAY: single concatenated column — no separate columns fed to CleanLab
    df['cleanlab_input'] = (
        df['Service'].fillna('') + ' | ' +
        df['keywords_list'].astype(str).fillna('') + ' | ' +
        df['summary'].fillna('')   # summary last = richest signal
    )

    # Cap max_features=15000 -> halves run time vs default 50k+ with minimal quality loss
    tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True)
    X_cl  = tfidf.fit_transform(df['cleanlab_input'])
    print(f"  TF-IDF matrix: {X_cl.shape}  (capped at 15k features for speed)")

    try:
        from cleanlab.multilabel_classification.label_quality_scores import (
            multilabel_label_quality_scores
        )
        clf_cl = OneVsRestClassifier(
            LogisticRegression(class_weight='balanced', max_iter=300, C=1.0), n_jobs=-1
        )
        print("  Computing 5-fold cross-val probabilities (approx 2 mins)...")
        pred_probs = cross_val_predict(clf_cl, X_cl, y, cv=5, method='predict_proba')
        quality    = multilabel_label_quality_scores(y, pred_probs)

        df['cleanlab_avg_quality'] = quality
        df['cleanlab_has_issue']   = quality < 0.7
        df['noise_score']          = (1.0 - quality.clip(0,1))

        cl_out = df[['case_reference','cleanlab_avg_quality','cleanlab_has_issue','noise_score']]
        cl_out.to_csv(CLEANLAB_CACHE, index=False)

        avg_quality = quality.mean()
        BASE_SMOOTH = round(min(0.15, max(0.02, 0.02 + (1.0 - avg_quality) * 0.25)), 3)

        print(f"  CleanLab done!")
        print(f"  Mean quality:           {quality.mean():.3f}")
        print(f"  Flagged noisy (< 0.7):  {(quality<0.7).sum()} ({(quality<0.7).mean()*100:.1f}%)")
        print(f"  Very noisy   (< 0.4):   {(quality<0.4).sum()} — consider dropping")
        print(f"  BASE_SMOOTH updated to: {BASE_SMOOTH}")
        print(f"  Cached to: {CLEANLAB_CACHE}")

    except ImportError:
        print("  cleanlab not installed — pip install cleanlab")
        print("  Using tag-count consistency as noise proxy instead")
        tc = y.sum(axis=1); exp = tc.mean(); std = tc.std() + 1e-6
        df['noise_score'] = (np.abs(tc - exp) / (std * 3)).clip(0, 0.5)

# Save cleaned dataframe after CleanLab
df.to_csv('saved_data/df_after_cleanlab.csv', index=False)
np.save('saved_data/y_labels.npy', y)
with open('saved_data/valid_tags.json','w') as f: json.dump(list(valid_tags), f)
print("Saved: saved_data/df_after_cleanlab.csv | y_labels.npy | valid_tags.json")


## Block 6 — Train / Val / Test Split
Split done **before** any augmentation to prevent data leakage. All splits saved to disk for reproducibility.

In [ ]:
idx = np.arange(len(df))
tr_idx, temp    = train_test_split(idx, test_size=0.2, random_state=42)
val_idx, te_idx = train_test_split(temp, test_size=0.5, random_state=42)

df_tr,  y_tr  = df.iloc[tr_idx].reset_index(drop=True), y[tr_idx]
df_val, y_val = df.iloc[val_idx].reset_index(drop=True), y[val_idx]
df_te,  y_te  = df.iloc[te_idx].reset_index(drop=True), y[te_idx]

df_tr.to_csv('saved_data/train_split.csv', index=False)
df_val.to_csv('saved_data/val_split.csv',  index=False)
df_te.to_csv('saved_data/test_split.csv',  index=False)
np.save('saved_data/y_tr.npy',  y_tr)
np.save('saved_data/y_val.npy', y_val)
np.save('saved_data/y_te.npy',  y_te)

print(f"Train: {len(df_tr)} | Val: {len(df_val)} | Test: {len(df_te)}")
print("Saved: train/val/test splits + label arrays to saved_data/")


## Block 7 — Imbalance Handling (3 Layers)
Applied in order on the **training set only**:
1. **Text augmentation** — copies of rare-tag complaints with minor word swaps/drops
2. **Undersampling** — removes 50% of rows that only have dominant tags
3. **WeightedRandomSampler** — rare-tag rows seen more per epoch

In [ ]:
# Layer 1: Text augmentation for rare tags
def augment_text(text):
    words = str(text).split()
    if len(words) < 6: return text
    op = random.choice(['swap', 'drop', 'repeat'])
    if op == 'swap' and len(words) > 3:
        i = random.randint(0, len(words)-2)
        words[i], words[i+1] = words[i+1], words[i]
    elif op == 'drop' and len(words) > 8:
        words.pop(random.randint(1, len(words)-2))
    elif op == 'repeat':
        mid = len(words)//2; words = words + words[mid:mid+6]
    return ' '.join(words)

if DO_AUGMENT:
    tag_counts_tr = y_tr.sum(axis=0)
    rare_idx      = np.where(tag_counts_tr < RARE_THRESHOLD)[0]
    aug_rows, aug_y = [], []
    for i in range(len(df_tr)):
        if y_tr[i, rare_idx].any():
            r = df_tr.iloc[i].copy()
            r['summary'] = augment_text(str(r.get('summary', '')))
            aug_rows.append(r); aug_y.append(y_tr[i])
    if aug_rows:
        df_tr = pd.concat([df_tr, pd.DataFrame(aug_rows)], ignore_index=True)
        y_tr  = np.vstack([y_tr, np.array(aug_y)])
        print(f"Layer 1: Augmented {len(aug_rows)} rare-tag rows -> train size: {len(df_tr)}")
else:
    print("Layer 1: Augmentation skipped (no rare tags below threshold)")

# Layer 2: Undersample dominant-only rows
if DO_UNDERSAMPLE:
    tag_freq_tr = y_tr.sum(axis=0) / len(y_tr)
    dom_set     = set(np.where(tag_freq_tr > DOMINANT_FREQ)[0])
    keep_i      = [i for i in range(len(df_tr))
                   if not set(np.where(y_tr[i]==1)[0]).issubset(dom_set)
                   or random.random() < 0.5]
    removed = len(df_tr) - len(keep_i)
    df_tr = df_tr.iloc[keep_i].reset_index(drop=True)
    y_tr  = y_tr[keep_i]
    print(f"Layer 2: Removed {removed} dominant-only rows -> train size: {len(df_tr)}")
else:
    print("Layer 2: Undersampling skipped (no dominant tags)")

# Layer 3: WeightedRandomSampler
tag_w = 1.0 / (y_tr.sum(axis=0) + 1e-6)
if 'sample_weight' in df_tr.columns:
    sw = df_tr['sample_weight'].fillna(1.0).values.astype(np.float32)
    sw = sw * (y_tr @ tag_w + 1e-6)
    print("Layer 3: Using existing sample_weight x rare-tag boost for sampler")
else:
    sw = (y_tr @ tag_w).astype(np.float32)
    print("Layer 3: Computed sample weights from inverse tag frequency")

sampler = WeightedRandomSampler(torch.tensor(sw, dtype=torch.float32), len(sw), replacement=True)
print(f"WeightedRandomSampler ready: {len(sw)} weights")


## Block 8 — Build Input Text, Dataset & DataLoaders
**Columns used:**
- `summary` — always (full complaint, richest signal)
- `Service` — always (product/service type)
- `tier2_consolidated_description` — always (semantic context)
- `keyword_group` — training: masked with `KW_DROPOUT` probability (forces model to read the complaint, not shortcut on keywords)

**Adaptive label smoothing:** noisy rows get more smoothing based on their `noise_score`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_text(row, is_training=False):
    parts = []
    for label, col in [('Service', 'Service'),
                        ('Complaint', 'summary'),
                        ('Context', 'tier2_consolidated_description')]:
        val = str(row.get(col, '')).strip()
        if val and val.lower() not in ('nan','none',''):
            parts.append(f"{label}: {val}")
    if not is_training or random.random() > KW_DROPOUT:
        kg = row.get('keyword_group_parsed', [])
        if isinstance(kg, list) and kg:
            parts.append(f"Categories: {', '.join(str(x) for x in kg)}")
    return ' [SEP] '.join(parts)

class ComplaintDataset(Dataset):
    def __init__(self, df_rows, labels, is_training=False):
        self.df = df_rows.reset_index(drop=True)
        self.labels = labels
        self.is_training = is_training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = tokenizer(build_text(row, self.is_training),
                        max_length=MAX_LEN, padding='max_length',
                        truncation=True, return_tensors='pt')
        lbl = torch.tensor(self.labels[idx], dtype=torch.float32)
        if self.is_training:
            ns     = float(row.get('noise_score', BASE_SMOOTH))
            smooth = min(0.20, max(0.01, BASE_SMOOTH + ns * 0.10))
            lbl    = lbl * (1 - smooth) + smooth * 0.5
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         lbl
        }

train_loader = DataLoader(ComplaintDataset(df_tr,  y_tr,  True),
                          batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader   = DataLoader(ComplaintDataset(df_val, y_val), batch_size=BATCH_SIZE, num_workers=0)
test_loader  = DataLoader(ComplaintDataset(df_te,  y_te),  batch_size=BATCH_SIZE, num_workers=0)

print(f"DataLoaders ready")
print(f"  Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"  MAX_LEN={MAX_LEN} | BATCH_SIZE={BATCH_SIZE} | KW_DROPOUT={KW_DROPOUT:.2f}")


## Block 9 — Model, Loss & Optimizer
- **DistilBERT** encoder with [CLS] pooling + dropout + linear head
- **Asymmetric Loss** with `gamma_neg` auto-scaled from imbalance ratio — penalises confident wrong predictions on dominant tags
- Swap `MODEL_NAME` to `bert-base-uncased` for slightly better performance if you have a GPU

In [ ]:
class ComplaintTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout    = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])   # [CLS] token
        return self.classifier(pooled)

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gn, self.gp, self.clip = gamma_neg, gamma_pos, clip

    def forward(self, logits, targets):
        p  = torch.sigmoid(logits)
        pn = (1 - p + self.clip).clamp(max=1)
        return (-(targets    * torch.log(p.clamp(1e-8))  * (1-p)**self.gp
                + (1-targets)* torch.log(pn.clamp(1e-8)) * p**self.gn)).mean()

model     = ComplaintTagger().to(DEVICE)
criterion = AsymmetricLoss(gamma_neg=GAMMA_NEG)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, total_steps // 10),
    num_training_steps=total_steps
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params:,} params  (66M = normal for DistilBERT)")
print(f"Loss: AsymmetricLoss(gamma_neg={GAMMA_NEG})  <- scaled from {imbalance_ratio:.0f}x imbalance")
print(f"Optimizer: AdamW | LR={LR} | Warmup={max(1, total_steps//10)} steps")


## Block 10 — Training Loop with Epoch Checkpoints
- Checkpoint saved after **every epoch** to `checkpoints/epoch_N.pt`
- Best model saved to `checkpoints/best_model.pt`
- Early stopping with patience auto-set by Block 3
- Training history saved to `saved_data/training_history.csv`

In [ ]:
def train_epoch(model, loader):
    model.train(); total = 0
    for b in tqdm(loader, desc="  Train", leave=False):
        optimizer.zero_grad()
        loss = criterion(
            model(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)),
            b['labels'].to(DEVICE))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total += loss.item()
    return total / len(loader)

def eval_epoch(model, loader):
    model.eval(); probs_all, labels_all = [], []
    with torch.no_grad():
        for b in tqdm(loader, desc="  Eval", leave=False):
            probs_all.append(
                torch.sigmoid(model(b['input_ids'].to(DEVICE),
                                    b['attention_mask'].to(DEVICE))).cpu().numpy())
            labels_all.append(b['labels'].numpy())
    return np.vstack(probs_all), np.vstack(labels_all)

print(f"Training: max {EPOCHS} epochs | patience={EARLY_STOP_PAT}\n")
best_f1, pat = 0.0, 0; history = []

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    tl     = train_epoch(model, train_loader)
    vp, vl = eval_epoch(model, val_loader)
    vl_h   = (vl >= 0.5).astype(int)
    macro  = f1_score(vl_h, (vp >= 0.5).astype(int), average='macro',  zero_division=0)
    micro  = f1_score(vl_h, (vp >= 0.5).astype(int), average='micro',  zero_division=0)
    history.append({'epoch': epoch+1, 'loss': round(tl,4),
                    'macro_f1': round(macro,4), 'micro_f1': round(micro,4)})
    print(f"  Loss: {tl:.4f}  |  Val Macro F1: {macro:.4f}  |  Val Micro F1: {micro:.4f}")

    # Save every epoch — lets you resume from any point
    torch.save({'epoch': epoch+1, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(), 'macro_f1': macro},
               f'checkpoints/epoch_{epoch+1}.pt')

    if macro > best_f1:
        best_f1 = macro
        torch.save(model.state_dict(), 'checkpoints/best_model.pt')
        print(f"  Best model saved (Macro F1: {best_f1:.4f})")
        pat = 0
    else:
        pat += 1
        if pat >= EARLY_STOP_PAT:
            print(f"  Early stop — best Macro F1: {best_f1:.4f}"); break

pd.DataFrame(history).to_csv('saved_data/training_history.csv', index=False)
print(f"\nTraining complete | Best Val Macro F1: {best_f1:.4f}")
print("Saved: checkpoints/best_model.pt | epoch_N.pt files | training_history.csv")


## Block 11 — Per-Tag Threshold Tuning
Finds the optimal decision threshold for each tag individually on the validation set.
- Dominant tags: threshold >= 0.65 (stop over-predicting)
- Rare tags: threshold <= 0.35 (catch more predictions)

In [ ]:
print("Loading best model and tuning per-tag thresholds...")
model.load_state_dict(torch.load('checkpoints/best_model.pt'))
vp, vl   = eval_epoch(model, val_loader)
vl_h     = (vl >= 0.5).astype(int)
tag_freq  = y_tr.sum(axis=0) / len(y_tr)
thresholds = []

for i in range(NUM_LABELS):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.05):
        f = f1_score(vl_h[:,i], (vp[:,i]>=t).astype(int), zero_division=0)
        if f > best_f: best_f, best_t = f, t
    if tag_freq[i] > DOMINANT_FREQ: best_t = max(best_t, 0.65)
    if tag_freq[i] < RARE_FREQ:     best_t = min(best_t, 0.35)
    thresholds.append(best_t)

thresholds = np.array(thresholds)
thr_df = pd.DataFrame({
    'tag': valid_tags,
    'train_freq_%': (tag_freq * 100).round(1),
    'threshold': thresholds,
    'category': ['dominant' if f > DOMINANT_FREQ
                 else 'rare' if f < RARE_FREQ
                 else 'normal' for f in tag_freq]
}).sort_values('train_freq_%', ascending=False)

np.save('saved_data/tag_thresholds.npy', thresholds)
thr_df.to_csv('saved_data/tag_info.csv', index=False)

print("Per-tag thresholds tuned and saved\n")
print(thr_df.to_string(index=False))


## Block 12 — Final Test Evaluation
Evaluates best model on the held-out test set using tuned thresholds.
**Macro F1 is your main metric** — treats all tags equally regardless of frequency.

In [ ]:
print("Running final test evaluation...")
tp, tl  = eval_epoch(model, test_loader)
tl_h    = (tl >= 0.5).astype(int)
preds   = (tp >= thresholds).astype(int)

macro_test = f1_score(tl_h, preds, average='macro',   zero_division=0)
micro_test = f1_score(tl_h, preds, average='micro',   zero_division=0)
samp_test  = f1_score(tl_h, preds, average='samples', zero_division=0)

print("=" * 55)
print(f"  Macro F1:    {macro_test:.4f}  <- main metric")
print(f"  Micro F1:    {micro_test:.4f}")
print(f"  Samples F1:  {samp_test:.4f}")
print("=" * 55)
print()
print(classification_report(tl_h, preds, target_names=valid_tags, zero_division=0))

pred_df = df_te[['case_reference']].copy()
for i, tag in enumerate(valid_tags):
    pred_df[f'pred_{tag}'] = preds[:, i]
    pred_df[f'prob_{tag}'] = tp[:, i].round(3)
pred_df.to_csv('saved_data/test_predictions.csv', index=False)
print("Saved: saved_data/test_predictions.csv")


## Block 13 — Human Review Export
Exports three batches of complaints to `human_review/complaints_for_review.csv`:

| Category | Why exported | How many |
|---|---|---|
| **Uncertain** | High entropy — model unsure, needs human label | ~33% of review budget |
| **Rare-tag predicted** | Verify model is correct on rare tag predictions | ~33% |
| **High-confidence** | Spot-check for systematic overconfidence | ~33% |

**How to use the CSV:**
1. Open `human_review/complaints_for_review.csv`
2. Read `summary` and `model_predicted_tags`
3. Fill `human_corrected_tags` with correct tier_2 tags (comma-separated)
4. Run Block 14 to merge corrections back

In [ ]:
def prediction_entropy(probs):
    p = np.clip(probs, 1e-9, 1-1e-9)
    return -(p * np.log(p) + (1-p) * np.log(1-p)).mean(axis=1)

entropy  = prediction_entropy(tp)
n_review = min(100, max(20, int(len(df_te) * 0.20)))
n_each   = max(5, n_review // 3)
base_cols = ['case_reference', 'summary', 'Service', 'tags_parsed']

def make_pred_tags(idx_list):
    return [', '.join(sorted([valid_tags[j] for j in np.where(preds[i]==1)[0]])) for i in idx_list]

# A) Most uncertain
uncertain_idx = np.argsort(entropy)[-n_each:]
df_unc = df_te.iloc[uncertain_idx][base_cols].copy()
df_unc['model_predicted_tags'] = make_pred_tags(uncertain_idx)
df_unc['model_confidence']     = entropy[uncertain_idx].round(3)
df_unc['review_reason']        = 'Uncertain — low model confidence'
df_unc['human_corrected_tags'] = ''

# B) Rare-tag predictions
rare_tag_idx_set = set(np.where(tag_freq < RARE_FREQ)[0])
rare_pred_rows   = [i for i in range(len(df_te))
                    if any(preds[i,j]==1 for j in rare_tag_idx_set)][:n_each]
df_rare = df_te.iloc[rare_pred_rows][base_cols].copy()
df_rare['model_predicted_tags'] = make_pred_tags(rare_pred_rows)
df_rare['model_confidence']     = [round(float(tp[i].max()), 3) for i in rare_pred_rows]
df_rare['review_reason']        = 'Rare tag predicted — please verify'
df_rare['human_corrected_tags'] = ''

# C) High confidence spot-check
high_conf_idx = np.argsort(-tp.max(axis=1))[:n_each]
df_hc = df_te.iloc[high_conf_idx][base_cols].copy()
df_hc['model_predicted_tags'] = make_pred_tags(high_conf_idx)
df_hc['model_confidence']     = tp[high_conf_idx].max(axis=1).round(3)
df_hc['review_reason']        = 'High confidence — spot-check for overconfidence'
df_hc['human_corrected_tags'] = ''

review_df = pd.concat([df_unc, df_rare, df_hc], ignore_index=True)              .drop_duplicates(subset=['case_reference'])
review_df.to_csv('human_review/complaints_for_review.csv', index=False)

print(f"Human review file exported: human_review/complaints_for_review.csv")
print(f"  Total rows:  {len(review_df)}")
print(f"  Uncertain:   {len(df_unc)}")
print(f"  Rare-tag:    {len(df_rare)}")
print(f"  High-conf:   {len(df_hc)}")
print()
print("NEXT STEPS:")
print("  1. Open human_review/complaints_for_review.csv")
print("  2. Fill 'human_corrected_tags' column (comma-separated tier_2 tags)")
print("  3. Run Block 14 to merge corrections and retrain")


## Block 14 — Feedback Loop (Run After Human Review)
Merges your human corrections back into the training set, then guides you to retrain.

In [ ]:
def apply_human_feedback(corrected_csv_path='human_review/complaints_for_review.csv'):
    """
    Run this AFTER filling 'human_corrected_tags' in the review CSV.
    Merges corrections into training data and saves updated set.

    Then go back to Block 2, set:
        DATA_FILE = 'saved_data/train_with_feedback.csv'
    and re-run from Block 2 onwards.
    """
    if not Path(corrected_csv_path).exists():
        print(f"File not found: {corrected_csv_path}")
        print("Complete Block 13 first"); return

    corrected = pd.read_csv(corrected_csv_path)
    corrected = corrected[corrected['human_corrected_tags'].notna() &
                          (corrected['human_corrected_tags'].str.strip() != '')]

    if corrected.empty:
        print("No corrections found — fill the 'human_corrected_tags' column first"); return

    print(f"Found {len(corrected)} human corrections — merging...")
    corrected['tags_parsed'] = corrected['human_corrected_tags'].apply(
        lambda x: [t.strip() for t in str(x).split(',') if t.strip()])

    df_feedback = pd.read_csv('saved_data/train_split.csv')
    df_feedback['tags_parsed'] = df_feedback['tags_parsed'].apply(parse_tags)
    count = 0

    for _, row in corrected.iterrows():
        match = df_feedback['case_reference'] == row['case_reference']
        if match.any():
            df_feedback.loc[match, 'tags_parsed'] = str(row['tags_parsed']); count += 1
        else:
            df_feedback = pd.concat([df_feedback, pd.DataFrame([row])], ignore_index=True)
            count += 1

    df_feedback.to_csv('saved_data/train_with_feedback.csv', index=False)
    print(f"Applied {count} corrections")
    print("Saved: saved_data/train_with_feedback.csv")
    print()
    print("TO RETRAIN WITH CORRECTIONS:")
    print("  1. Go to Block 2")
    print("  2. Set DATA_FILE = 'saved_data/train_with_feedback.csv'")
    print("  3. Re-run all blocks from Block 2 onwards")

apply_human_feedback()


## Block 15 — Inference on New Complaints
Use the trained model to predict tier_2 tags on any new complaint row.

In [ ]:
def predict_tags(row_dict, top_k=None):
    """
    Predict tier_2 tags for a single complaint.

    Parameters:
        row_dict : dict — keys: summary, Service,
                          tier2_consolidated_description (optional),
                          keyword_group_parsed (optional list)
        top_k    : int — if no tag passes threshold, return top_k by probability

    Returns:
        tags : list of predicted tag strings
        conf : dict of {tag: probability}
    """
    row  = pd.Series(row_dict)
    text = build_text(row, is_training=False)
    enc  = tokenizer(text, max_length=MAX_LEN, padding='max_length',
                     truncation=True, return_tensors='pt')
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(
            model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
        ).cpu().numpy()[0]

    idx  = np.where(probs >= thresholds)[0]
    if top_k and len(idx) == 0:
        idx = np.argsort(probs)[-top_k:]

    tags = sorted([valid_tags[i] for i in idx])
    conf = {valid_tags[i]: round(float(probs[i]), 3)
            for i in sorted(idx, key=lambda x: -probs[x])}
    return tags, conf


# Example usage
example_complaint = {
    'Service':                        'Vehicle loan or lease',
    'summary':                        'Account breach on my vehicle loan. I disputed immediately but their investigation was inadequate.',
    'tier2_consolidated_description': 'Account breach Fraudulent charges',
    'keyword_group_parsed':           ['Fraud and security', 'Consumer protection'],
}

tags, conf = predict_tags(example_complaint, top_k=3)
print(f"Predicted tags: {tags}")
print()
print("Confidence scores:")
for tag, score in conf.items():
    bar = '#' * int(score * 20)
    print(f"  {tag:<40} {score:.3f}  {bar}")

print()
print("Batch prediction on a new CSV:")
print("  new_df = pd.read_csv('new_complaints.csv')")
print("  results = new_df.apply(lambda r: predict_tags(r.to_dict()), axis=1)")
print("  new_df['predicted_tags'], new_df['confidence'] = zip(*results)")
